In [ ]:
cols_to_drop = [
    'op_id','subject_id','hadm_id','opdate'
]
df = df.drop(columns=cols_to_drop)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['mortality'])
y = df['mortality']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

In [ ]:
# y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]
y_pred = (y_prob > 0.2).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred = (y_prob > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

In [ ]:
y_prob_rf = rf.predict_proba(X_test)[:,1]
y_pred_rf = (y_prob_rf > 0.4).astype(int)

print(classification_report(y_test, y_pred_rf))

In [ ]:
for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_pred_rf = (y_prob_rf > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred_rf))

In [ ]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [ ]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [ ]:
model_sm = LogisticRegression(max_iter=1000)
model_sm.fit(X_train_sm, y_train_sm)

In [ ]:
y_prob_sm = model_sm.predict_proba(X_test)[:,1]
y_pred_sm = (y_prob_sm > 0.5).astype(int)

print(classification_report(y_test, y_pred_sm))

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight= (len(y_train) / sum(y_train)),  # handles imbalance
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

In [ ]:
y_prob_xgb = xgb.predict_proba(X_test)[:,1]
y_pred_xgb = (y_prob_xgb > 0.3).astype(int)

from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

In [ ]:
import shap

explainer = shap.Explainer(xgb)
shap_values = explainer(X_test)

In [ ]:
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
for t in [0.05, 0.1, 0.2]:
    y_pred_xgb = (y_prob_xgb > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred_xgb))

In [ ]:
import joblib

joblib.dump(model_sm, "model.pkl")

In [ ]:
joblib.dump(X_train.columns, "features.pkl")